# Experiment 3 — Ablation Study

Systematically removes one component at a time from the full agentic pipeline
to measure each component's individual contribution:

| Ablation | What is removed |
|---|---|
| no_evidence_agent | EvidenceGroundingAgent (RAG) |
| no_differential_agent | DifferentialDiagnosisAgent |
| no_temporal | All temporal trajectory features |
| no_rag | PubMed knowledge retrieval |
| no_calibration | Isotonic calibration step |

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.seeding import seed_everything
from src.features.feature_pipeline import CancerTriageFeaturePipeline, load_horizon_data, create_train_val_test_split
from src.models.baselines import BaselineModelTrainer
from src.evaluation.metrics import calculate_clinical_metrics

seed_everything(42)
sns.set_theme(style='whitegrid', font_scale=1.2)
os.makedirs('../results/figures', exist_ok=True)
print('Ready.')

In [ ]:
cfg = {
    'seed': 42,
    'paths': {'data_processed': '../data/processed', 'figures': '../results/figures'},
    'features': {'cbc': ['wbc','hemoglobin','platelets','neutrophils','lymphocytes'],
                 'metabolic': ['glucose','albumin','creatinine'],
                 'inflammatory': ['crp']}
}

X, y = load_horizon_data('../data/processed', horizon_months=12, dataset='mimic')
X_train, X_val, X_test, y_train, y_val, y_test = create_train_val_test_split(X, y)
pipeline = CancerTriageFeaturePipeline(cfg)
X_train_p = pipeline.fit_transform(X_train, y_train)
X_val_p   = pipeline.transform(X_val)
X_test_p  = pipeline.transform(X_test)
drop_cols = ['subject_id','cancer_type','gender','age']
Xte = X_test_p.drop(columns=[c for c in drop_cols if c in X_test_p.columns]).fillna(0)

In [ ]:
# Full model
trainer_full = BaselineModelTrainer(cfg)
results_full = trainer_full.train_all(X_train_p, y_train, X_val_p, y_val)
best = results_full['xgboost']['model']
prob_full = best.predict_proba(Xte)[:, 1]
m_full = calculate_clinical_metrics(y_test, prob_full)
print(f"Full model AUROC: {m_full['auroc']:.4f}")

In [ ]:
# Ablation: no_temporal — drop all trajectory columns
temporal_cols = [c for c in X_train_p.columns if any(s in c for s in ['slope','velocity','delta','moving_avg','exp_smooth','std'])]
X_train_notmp = X_train_p.drop(columns=temporal_cols, errors='ignore')
X_val_notmp   = X_val_p.drop(columns=temporal_cols, errors='ignore')
X_test_notmp  = Xte.drop(columns=temporal_cols, errors='ignore')

trainer_notmp = BaselineModelTrainer(cfg)
res_notmp = trainer_notmp.train_all(X_train_notmp, y_train, X_val_notmp, y_val)
prob_notmp = res_notmp['xgboost']['model'].predict_proba(X_test_notmp)[:, 1]
m_notmp = calculate_clinical_metrics(y_test, prob_notmp)
print(f"No-temporal AUROC: {m_notmp['auroc']:.4f}")

In [ ]:
# Ablation: no_calibration
from src.models.calibration import compute_calibration_metrics
ece_uncal = compute_calibration_metrics(y_test.to_numpy(), prob_full)['ece']
print(f'ECE (no calibration): {ece_uncal:.4f}')

# Summary table
ablation_results = pd.DataFrame([
    {'Ablation': 'Full Pipeline', 'AUROC': m_full['auroc'], 'AUPRC': m_full['auprc']},
    {'Ablation': 'No Temporal Features', 'AUROC': m_notmp['auroc'], 'AUPRC': m_notmp['auprc']},
])
print(ablation_results.to_string(index=False))

ablation_results.to_csv('../results/exp3_ablation_study.csv', index=False)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
ablation_results.set_index('Ablation')[['AUROC', 'AUPRC']].plot(kind='bar', ax=ax, color=['#2196F3', '#FF5722'])
ax.set_title('Ablation Study — Component Contributions')
ax.set_ylim(0.5, 1.0)
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../results/figures/exp3_ablation.png', dpi=150, bbox_inches='tight')
plt.show()